In [ ]:
!pip install transformers torch sqlite-web

In [ ]:


import sqlite3

conn = sqlite3.connect("company.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE employees(
id INTEGER PRIMARY KEY,
name TEXT,
salary INTEGER,
city TEXT
)
""")

cursor.execute("INSERT INTO employees VALUES (1,'Arun',60000,'Chennai')")
cursor.execute("INSERT INTO employees VALUES (2,'Priya',45000,'Madurai')")
cursor.execute("INSERT INTO employees VALUES (3,'Rahul',70000,'Chennai')")
cursor.execute("INSERT INTO employees VALUES (4,'Anu',30000,'Trichy')")

conn.commit()
conn.close()

print("Database created")

In [ ]:
from transformers import pipeline

generator = pipeline("text-generation", model="distilgpt2")

In [ ]:

def generate_sql(question):

    question = question.lower()

    if "all employees" in question:
        return "SELECT * FROM employees"

    elif "salary greater than 50000" in question:
        return "SELECT * FROM employees WHERE salary > 50000"

    elif "employees from chennai" in question:
        return "SELECT * FROM employees WHERE city='Chennai'"

    elif "employees from madurai" in question:
        return "SELECT * FROM employees WHERE city='Madurai'"

    else:
        return None

In [ ]:

import sqlite3

question = input("Enter your question: ")

query = generate_sql(question)

print("Generated SQL:", query)

conn = sqlite3.connect("company.db")
cursor = conn.cursor()

cursor.execute(query)

rows = cursor.fetchall()

for r in rows:
    print(r)

conn.close()

In [ ]:
!pip install gradio

In [ ]:
import gradio as gr
import sqlite3

def run_query(question):

    query = generate_sql(question)

    conn = sqlite3.connect("company.db")
    cursor = conn.cursor()

    cursor.execute(query)
    rows = cursor.fetchall()

    conn.close()

    result = f"Generated SQL:\n{query}\n\nResult:\n{rows}"
    return result


css = """
body {background-color:#0f172a;}
.gradio-container {background-color:#0f172a; color:white}
textarea {font-size:18px !important}
"""

with gr.Blocks(css=css) as demo:

    gr.Markdown("# SQL Query Generator")

    question = gr.Textbox(
        label="Enter your Question",
        placeholder="Example: show all employees",
        lines=2
    )

    btn = gr.Button("Generate SQL")

    output = gr.Textbox(
        label="Output",
        lines=12
    )

    btn.click(run_query, inputs=question, outputs=output)

demo.launch()